In [ ]:
# =============================================================
# Divisione Construction Set (CS) in TRS (80%) e VS (20%)
# con bilanciamento 50/50 tra le classi (Diverticolite / Tumore)
# =============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from google.colab import drive

In [ ]:
# 1️⃣ Monta Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =============================================================
# Divisione Construction Set (CS) in TRS (80%) e VS (20%)
# mantenendo la distribuzione originale delle classi
# (senza undersampling) e forzando il caso 104 nel VS
# =============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from google.colab import drive

OUTLIER_ID = 104  # <--- ID da mettere solo nel VS

# 1️⃣ Monta Google Drive
drive.mount('/content/drive')

# 2️⃣ Percorsi dei file (rimasti invariati)
diagnosi_path = '/content/drive/MyDrive/DSH - Lab/Diagnosi.xlsx'   # File Excel esportato
cs_ids_path   = '/content/drive/MyDrive/DSH - Lab/Lab1/CS.txt'     # File di testo con gli ID

# 3️⃣ Carica file delle diagnosi
df = pd.read_excel(diagnosi_path)
target_col = 'DEFINITIVE DIAGNOSIS '  # attenzione allo spazio finale!

# 4️⃣ Carica lista ID dal file .txt (lettura robusta)
cs_ids = []
with open(cs_ids_path, 'r', encoding='utf-8-sig') as f:
    for line in f:
        line = line.strip()
        if line.isdigit():
            cs_ids.append(int(line))

print(f"✅ Importati {len(cs_ids)} ID dal file CS.txt")

# 5️⃣ Pulizia e filtraggio
df['CASE NUMBER'] = pd.to_numeric(df['CASE NUMBER'], errors='coerce')
df = df.dropna(subset=['CASE NUMBER'])
df['CASE NUMBER'] = df['CASE NUMBER'].astype(int)
df_cs = df[df['CASE NUMBER'].isin(cs_ids)].copy()

print(f"📂 Totale pazienti nel CS trovati nel file Diagnosi: {len(df_cs)}")

# 🔹 Separiamo l'outlier (104) dal resto
df_outlier   = df_cs[df_cs['CASE NUMBER'] == OUTLIER_ID].copy()
df_no_outlier = df_cs[df_cs['CASE NUMBER'] != OUTLIER_ID].copy()

if df_outlier.empty:
    print(f"⚠️ Attenzione: OUTLIER_ID={OUTLIER_ID} non trovato nel CS!")
else:
    print(f"✅ Outlier {OUTLIER_ID} trovato e verrà messo nel VS")

# 6️⃣ Shuffle del dataset SENZA l'outlier
df_bal = df_no_outlier.sample(frac=1, random_state=42).reset_index(drop=True)

# 7️⃣ Divisione 80/20 stratificata (solo sui non-outlier)
X = df_bal[['CASE NUMBER']]
y = df_bal[target_col]

X_trs, X_vs, y_trs_tmp, y_vs_tmp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

trs_ids = X_trs['CASE NUMBER'].tolist()
vs_ids  = X_vs['CASE NUMBER'].tolist()

# 8️⃣ Forza l'outlier nel VS ed escludilo dal TRS
if not df_outlier.empty:
    out_id = int(df_outlier['CASE NUMBER'].iloc[0])

    # togli da TRS se per qualche motivo ci fosse
    if out_id in trs_ids:
        trs_ids.remove(out_id)

    # aggiungi a VS se non già presente
    if out_id not in vs_ids:
        vs_ids.append(out_id)

# 9️⃣ Ricalcolo delle serie y_trs e y_vs sulla base degli ID finali
y_trs = df_cs[df_cs['CASE NUMBER'].isin(trs_ids)][target_col]
y_vs  = df_cs[df_cs['CASE NUMBER'].isin(vs_ids)][target_col]

#  🔍 Controllo finale: l'outlier è solo nel VS?
if OUTLIER_ID in trs_ids:
    print(f"❌ ERRORE: {OUTLIER_ID} è ancora nel TRS!")
else:
    print(f"✅ Conferma: {OUTLIER_ID} NON è nel TRS")

if OUTLIER_ID in vs_ids:
    print(f"✅ Conferma: {OUTLIER_ID} è nel VS")
else:
    print(f"⚠️ Attenzione: {OUTLIER_ID} NON è nel VS")

# 🔟 Bilanciamento e conteggi
print("\n📊 Distribuzione percentuale classi:")
print("TRS →")
print(y_trs.value_counts(normalize=True).sort_index() * 100)
print("\nVS →")
print(y_vs.value_counts(normalize=True).sort_index() * 100)

print("\n📈 Conteggi assoluti:")
print("TRS →")
print(y_trs.value_counts().sort_index())
print("\nVS →")
print(y_vs.value_counts().sort_index())

# 1️⃣1️⃣ Salvataggio su Drive
trs_path = '/content/drive/MyDrive/DSH - Lab/TRS_IDs_80.csv'
vs_path  = '/content/drive/MyDrive/DSH - Lab/VS_IDs_20.csv'

pd.DataFrame({'CASE NUMBER': trs_ids}).to_csv(trs_path, index=False)
pd.DataFrame({'CASE NUMBER': vs_ids}).to_csv(vs_path, index=False)

print(f"\n✅ File salvati in:\n  - {trs_path}\n  - {vs_path}")


Mounted at /content/drive
✅ Importati 106 ID dal file CS.txt
📂 Totale pazienti nel CS trovati nel file Diagnosi: 106
✅ Outlier 104 trovato e verrà messo nel VS
✅ Conferma: 104 NON è nel TRS
✅ Conferma: 104 è nel VS

📊 Distribuzione percentuale classi:
TRS →
DEFINITIVE DIAGNOSIS 
0    55.952381
1    44.047619
Name: proportion, dtype: float64

VS →
DEFINITIVE DIAGNOSIS 
0    54.545455
1    45.454545
Name: proportion, dtype: float64

📈 Conteggi assoluti:
TRS →
DEFINITIVE DIAGNOSIS 
0    47
1    37
Name: count, dtype: int64

VS →
DEFINITIVE DIAGNOSIS 
0    12
1    10
Name: count, dtype: int64

✅ File salvati in:
  - /content/drive/MyDrive/DSH - Lab/TRS_IDs_80.csv
  - /content/drive/MyDrive/DSH - Lab/VS_IDs_20.csv
